In [ ]:
# TAREA: LSTM vs GRU + INTRODUCCIÓN A MODELOS DE LENGUAJE

# INSTRUCCIONES GENERALES:
# 1) Ejecuta el notebook completo.
# 2) Modifica el dataset del bloque A en al menos 5 elementos.
# 3) Modifica el corpus del bloque B en al menos 5 elementos.
# 4) Compara resultados e interpreta.
#--------------------------------------------------------------
# CELDA 1: IMPORTS Y CONFIGURACIÓN

# numpy para operaciones numéricas
import numpy as np

# pandas para tablas
import pandas as pd

# matplotlib para gráficas
import matplotlib.pyplot as plt

# train_test_split para separar train y test
from sklearn.model_selection import train_test_split

# tensorflow como framework principal
import tensorflow as tf

# Tokenizer para convertir texto a secuencias de enteros
from tensorflow.keras.preprocessing.text import Tokenizer

# pad_sequences para igualar longitudes
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sequential para construir modelos capa por capa
from tensorflow.keras.models import Sequential

# Capas que usaremos
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Input

# to_categorical para convertir etiquetas enteras a one-hot
from tensorflow.keras.utils import to_categorical

# Adam como optimizador
from tensorflow.keras.optimizers import Adam

# Fijamos semillas para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

In [ ]:
# BLOQUE A: CLASIFICACIÓN DE TEXTO CON LSTM Y GRU

# Deberas modificar al menos 5 elementos del dataset:
# - sujetos
# - verbos
# - adjetivos positivos
# - adjetivos negativos
# - complementos


# Sujetos
sujetos = [
    "la clase",
    "el curso",
    "la presentacion",
    "el sistema",
    "el modelo",
    "la red",
    "la explicacion",
    "el resultado",
    "la practica",
    "la sesion"
]

# Verbos
verbos = [
    "fue",
    "estuvo",
    "resulto",
    "parecio"
]

# Adjetivos positivos
positivos = [
    "excelente",
    "clara",
    "util",
    "ordenada",
    "brillante",
    "precisa",
    "interesante",
    "estable",
    "correcta",
    "eficiente"
]

# Adjetivos negativos
negativos = [
    "terrible",
    "confusa",
    "mala",
    "aburrida",
    "inestable",
    "incorrecta",
    "deficiente",
    "pesada",
    "desordenada",
    "pobre"
]

# Complementos
complementos = [
    "para el grupo",
    "durante la sesion",
    "en esta practica",
    "para el analisis",
    "segun los resultados",
    "en el laboratorio",
    "durante la clase",
    "para este problema",
    "en la evaluacion final",
    "segun el experimento"
]

# Listas vacías para frases y etiquetas
frases = []
etiquetas = []

# Importamos random
import random
random.seed(42)

# Generamos ejemplos positivos
for _ in range(300):
    sujeto = random.choice(sujetos)
    verbo = random.choice(verbos)
    adj = random.choice(positivos)
    comp = random.choice(complementos)
    frase = f"{sujeto} {verbo} {adj} {comp}"
    frases.append(frase)
    etiquetas.append(1)

# Generamos ejemplos negativos
for _ in range(300):
    sujeto = random.choice(sujetos)
    verbo = random.choice(verbos)
    adj = random.choice(negativos)
    comp = random.choice(complementos)
    frase = f"{sujeto} {verbo} {adj} {comp}"
    frases.append(frase)
    etiquetas.append(0)

# Convertimos a DataFrame
df = pd.DataFrame({
    "frase": frases,
    "etiqueta": etiquetas
})

# Mezclamos el dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Mostramos ejemplos
print(df.head())
print("\nTotal de ejemplos:", len(df))
print("\nBalance de clases:")
print(df["etiqueta"].value_counts())

In [ ]:
# CELDA 3: TRAIN / TEST SPLIT PARA CLASIFICACIÓN

# Extraemos frases
X = df["frase"].tolist()

# Extraemos etiquetas
y = df["etiqueta"].tolist()

# Dividimos train y test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Número de ejemplos de train:", len(X_train))
print("Número de ejemplos de test :", len(X_test))

In [ ]:
# CELDA 4: TOKENIZACIÓN Y PADDING PARA CLASIFICACIÓN

# Creamos tokenizer
tokenizer = Tokenizer(oov_token="<OOV>")

# Aprendemos vocabulario
tokenizer.fit_on_texts(X_train)

# Diccionario palabra -> índice
word_index = tokenizer.word_index

# Convertimos frases a secuencias numéricas
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Longitud máxima
max_len = max(len(seq) for seq in X_train_seq)

# Aplicamos padding
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post")

# Tamaño del vocabulario
vocab_size = len(word_index) + 1

print("Tamaño del vocabulario:", len(word_index))
print("Longitud máxima:", max_len)
print("Forma X_train_pad:", X_train_pad.shape)
print("Forma X_test_pad :", X_test_pad.shape)

In [ ]:
# CELDA 5: MODELO LSTM

# Dimensión del embedding
embedding_dim = 16

# Creamos modelo LSTM
modelo_lstm = Sequential()

# Definimos entrada
modelo_lstm.add(Input(shape=(max_len,)))

# Embedding
modelo_lstm.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True
    )
)

# Capa LSTM
modelo_lstm.add(
    LSTM(
        16,
        recurrent_dropout=0.2
    )
)

# Salida binaria
modelo_lstm.add(Dense(1, activation="sigmoid"))

# Compilamos
modelo_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

modelo_lstm.summary()

In [ ]:
# CELDA 6: MODELO GRU

# Creamos modelo GRU
modelo_gru = Sequential()

# Definimos entrada
modelo_gru.add(Input(shape=(max_len,)))

# Embedding
modelo_gru.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True
    )
)

# Capa GRU
modelo_gru.add(
    GRU(
        16,
        recurrent_dropout=0.2
    )
)

# Salida binaria
modelo_gru.add(Dense(1, activation="sigmoid"))

# Compilamos
modelo_gru.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

modelo_gru.summary()

In [ ]:
# CELDA 7: ENTRENAMIENTO DE LSTM Y GRU

# Entrenamos LSTM
hist_lstm = modelo_lstm.fit(
    X_train_pad,
    np.array(y_train),
    epochs=20,
    validation_split=0.2,
    verbose=0
)

# Entrenamos GRU
hist_gru = modelo_gru.fit(
    X_train_pad,
    np.array(y_train),
    epochs=20,
    validation_split=0.2,
    verbose=0
)

print("Entrenamiento completado.")

In [ ]:
# CELDA 8: GRÁFICAS COMPARATIVAS

# Accuracy
plt.figure(figsize=(12, 5))
plt.plot(hist_lstm.history["val_accuracy"], label="LSTM val_accuracy")
plt.plot(hist_gru.history["val_accuracy"], label="GRU val_accuracy")
plt.title("Comparación de accuracy en validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# Loss
plt.figure(figsize=(12, 5))
plt.plot(hist_lstm.history["val_loss"], label="LSTM val_loss")
plt.plot(hist_gru.history["val_loss"], label="GRU val_loss")
plt.title("Comparación de loss en validación")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# CELDA 9: EVALUACIÓN FINAL EN TEST

# Evaluamos LSTM
resultado_lstm = modelo_lstm.evaluate(X_test_pad, np.array(y_test), verbose=0)

# Evaluamos GRU
resultado_gru = modelo_gru.evaluate(X_test_pad, np.array(y_test), verbose=0)

print("Resultados en test:")
print(f"LSTM -> Loss: {resultado_lstm[0]:.4f} | Accuracy: {resultado_lstm[1]:.4f}")
print(f"GRU  -> Loss: {resultado_gru[0]:.4f} | Accuracy: {resultado_gru[1]:.4f}")

In [ ]:
# CELDA 10: FRASES NUEVAS

# Puedes agregar aquí nuevas frases para probar generalización.

frases_nuevas = [
    "la clase fue excelente durante la sesion",
    "el curso estuvo terrible para el grupo",
    "la explicacion resulto clara en el laboratorio",
    "el sistema parecio inestable segun los resultados"
]

# Convertimos a secuencias
seq_nuevas = tokenizer.texts_to_sequences(frases_nuevas)

# Aplicamos padding
pad_nuevas = pad_sequences(seq_nuevas, maxlen=max_len, padding="post")

# Predicciones
pred_lstm = modelo_lstm.predict(pad_nuevas, verbose=0)
pred_gru = modelo_gru.predict(pad_nuevas, verbose=0)

print("Predicciones sobre frases nuevas:\n")
for i, frase in enumerate(frases_nuevas):
    print(f"Frase: {frase}")
    print(f"  LSTM: {pred_lstm[i][0]:.4f}")
    print(f"  GRU : {pred_gru[i][0]:.4f}")
    print("-" * 70)

In [ ]:
# BLOQUE B: INTRODUCCIÓN A MODELOS DE LENGUAJE

# Construye un mini dataset con ventana deslizante
# para predecir la siguiente palabra.
# Deberas modificar al menos 5 elementos:
# - corpus
# - tamaño de ventana
# - prompts
# - número de épocas

corpus = [
    "la inteligencia artificial aprende patrones",
    "las redes neuronales procesan secuencias",
    "las gru simplifican la memoria recurrente",
    "las lstm controlan el flujo de informacion",
    "el siguiente token depende del contexto",
    "la salida softmax produce probabilidades",
    "el modelo de lenguaje predice la siguiente palabra"
]

print("Corpus de ejemplo:")
for frase in corpus:
    print("-", frase)

In [ ]:
# CELDA 12: TOKENIZACIÓN DEL CORPUS Y VENTANA DESLIZANTE

# Creamos tokenizer para el corpus
tokenizer_lm = Tokenizer()

# Aprendemos vocabulario
tokenizer_lm.fit_on_texts(corpus)

# Tamaño del vocabulario
total_words = len(tokenizer_lm.word_index) + 1

# Lista donde guardaremos secuencias
input_sequences = []

# Construimos ejemplos con ventana deslizante
for frase in corpus:
    token_list = tokenizer_lm.texts_to_sequences([frase])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i + 1]
        input_sequences.append(n_gram_sequence)

# Longitud máxima
max_sequence_len = max(len(seq) for seq in input_sequences)

# Hacemos padding
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre")
)

# X = historial
X_lm = input_sequences[:, :-1]

# y = siguiente token
y_lm = input_sequences[:, -1]

# One-hot para la salida softmax
y_lm = to_categorical(y_lm, num_classes=total_words)

print("Forma de X_lm:", X_lm.shape)
print("Forma de y_lm:", y_lm.shape)
print("Vocabulario total:", total_words)

In [ ]:
# CELDA 13: MODELO DE LENGUAJE CON GRU

modelo_lm = Sequential()

# Entrada
modelo_lm.add(Input(shape=(max_sequence_len - 1,)))

# Embedding
modelo_lm.add(Embedding(input_dim=total_words, output_dim=32))

# GRU
modelo_lm.add(GRU(32))

# Salida softmax
modelo_lm.add(Dense(total_words, activation="softmax"))

# Compilamos
modelo_lm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

modelo_lm.summary()

In [ ]:
# CELDA 14: ENTRENAMIENTO DEL MODELO DE LENGUAJE

hist_lm = modelo_lm.fit(
    X_lm,
    y_lm,
    epochs=150,
    verbose=0
)

print("Entrenamiento del modelo de lenguaje completado.")

In [ ]:
# CELDA 15: FUNCIÓN SIMPLE DE GENERACIÓN

def generar_texto(modelo, tokenizer, seed_text, max_sequence_len, n_palabras=5):
    # Repetimos el proceso de generación varias veces
    for _ in range(n_palabras):
        # Convertimos el texto semilla a secuencia
        token_list = tokenizer.texts_to_sequences([seed_text])[0]

        # Aplicamos padding
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding="pre")

        # Predecimos distribución
        predicted = modelo.predict(token_list, verbose=0)

        # Elegimos la palabra más probable
        predicted_index = np.argmax(predicted, axis=1)[0]

        # Buscamos qué palabra corresponde a ese índice
        output_word = ""
        for palabra, indice in tokenizer.word_index.items():
            if indice == predicted_index:
                output_word = palabra
                break

        # Si no se encuentra palabra, detenemos
        if output_word == "":
            break

        # Agregamos la palabra al texto
        seed_text += " " + output_word

    return seed_text

In [ ]:
# CELDA 16: PROBAR EL MODELO DE LENGUAJE

prompts = [
    "la inteligencia",
    "las gru",
    "el siguiente"
]

for prompt in prompts:
    generado = generar_texto(modelo_lm, tokenizer_lm, prompt, max_sequence_len, n_palabras=5)
    print("Prompt:", prompt)
    print("Texto generado:", generado)
    print("-" * 70)

In [ ]:
# CONCLUSIÓN:
# Escribe aquí una conclusión breve donde respondas:
# 1. ¿Qué diferencias viste entre LSTM y GRU?
# 2. ¿La GRU logró rendimiento parecido a la LSTM?
# 3. ¿Qué hace la ventana deslizante?
# 4. ¿Qué función cumple softmax en el modelo de lenguaje?
# 5. ¿En qué cambia el problema cuando pasamos de clasificar a generar?

print("Escribe aquí tu conclusión en una celda Markdown o comentario.")